# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
Source schema (Croissant): https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure mlcroissant is available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(url)
# Access metadata
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

This step provides insight into the structure of the dataset, including which record sets exist and what fields they contain.

In [ ]:
# List all available record sets and their fields by @id
record_sets = dataset.record_sets
print("Available record sets and their fields:")
for rs in record_sets:
    print(f'RecordSet @id: {rs["@id"]}')
    print(f'  Name: {rs.get("name", "(no name)")}')
    fields = rs.get('fields', [])
    for field in fields:
        print(f'    Field @id: {field["@id"]}, Name: {field.get("name", "")}, Type: {field.get("dataType", "")}', )
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Refer to each entity by its `@id`.

Here, we extract all available record sets defined in the schema and create one DataFrame per record set.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs["@id"] for rs in dataset.record_sets]

# Load each as a DataFrame
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Show columns for the first record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns for RecordSet {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    print(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping.

Choose numeric and categorical fields by their `@id` as present in the schema.

In [ ]:
import numpy as np

# Identify a numeric field for EDA
# For this demo, suppose 'GAD7_score' and 'village' are actual column keys in the dataset
# You must reference real @id values. Let's assume these:
record_set_id = first_rs_id  # uses the first RecordSet found
numeric_field_id = 'GAD7_score'      # placeholder: update with real @id if available
group_field_id = 'village'           # placeholder: update with real @id if available

df = dataframes[record_set_id]
# Check if the fields exist
if numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - np.mean(filtered_df[numeric_field_id])) / np.std(filtered_df[numeric_field_id])
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field '{numeric_field_id}' not found in DataFrame columns: {df.columns.tolist()}.")

## 5. Visualization
Visualize distributions and relationships between fields. Use fields by their `@id`.

Here, we'll plot the distribution of the numeric variable and its relationship with a demographic group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step loading, exploration, and basic processing of the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the Croissant schema and `mlcroissant` API.
Key findings and workflow steps:
- Dataset structure was reviewed by listing record sets and fields by `@id`.
- Survey responses were loaded into DataFrames for analysis.
- Numeric fields were filtered, normalized, and grouped by demographic variables referenced by their `@id`s.
- Plots visualized both distribution and relationships as examples for further research.

For full details or advanced analysis, refer to the actual dataset documentation and adjust field references as needed according to their `@id`.